# Лекция: Построение таблиц частот в Python

**Дисциплина:** Введение в анализ больших данных

Тема лекции — перевод непрерывных (или порядковых) чисел в **категории**, подсчёт частот и визуализация:
- разбиение на интервалы (`pd.cut`);
- абсолютные и относительные частоты (`value_counts`);
- объединение таблиц (`pd.concat`);
- столбчатые и круговые диаграммы.

Примеры ниже используют **возрасты сотрудников двух отделов** и свои границы интервалов. Они **не совпадают** с лабораторным заданием: цель — освоить методы, задание выполните самостоятельно на своих данных.


## 0. Импорт и демо-данные


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
np.random.seed(42)

# Демо: возраст сотрудников (лет) в двух отделах
dept_A = np.array([
    22, 25, 28, 31, 33, 35, 36, 38, 40, 41,
    42, 44, 45, 47, 48, 50, 52, 55, 58, 61,
    24, 27, 29, 34, 37, 39, 43, 46, 49, 53,
])
dept_B = np.array([
    21, 23, 26, 28, 30, 32, 34, 36, 38, 40,
    42, 45, 48, 51, 54, 57, 60, 63, 25, 29,
    33, 37, 41, 44, 47, 50, 53, 56, 59, 62,
    27, 35, 43, 49, 55,
])

print(f"Отдел A: n={len(dept_A)}, min={dept_A.min()}, max={dept_A.max()}")
print(f"Отдел B: n={len(dept_B)}, min={dept_B.min()}, max={dept_B.max()}")


---
## 1. Разбиение на категории: `pd.cut`

Числовой вектор «нарезается» на интервалы. Нужны:
- **bins** — границы интервалов (или число равных интервалов);
- **labels** — имена категорий;
- **include_lowest=True** — чтобы минимальное значение не стало `NaN`.

**Пример шкалы «возрастные группы»:**

| Группа | Возраст |
|--------|---------|
| junior | 18–29 |
| mid | 30–44 |
| senior | 45–54 |
| lead | 55–70 |


In [ ]:
bins_age = [17.9, 29, 44, 54, 70]
labels_age = ["junior", "mid", "senior", "lead"]

cat_A = pd.cut(dept_A, bins=bins_age, labels=labels_age, include_lowest=True)
cat_B = pd.cut(dept_B, bins=bins_age, labels=labels_age, include_lowest=True)

print("Отдел A:")
print(cat_A.value_counts().sort_index())
print("\nОтдел B:")
print(cat_B.value_counts().sort_index())


### Другой способ: равные интервалы или квантили

Можно указать число интервалов или границы по квантилям.


In [ ]:
# Три равных интервала по диапазону
eq = pd.cut(dept_A, bins=3, labels=["low", "med", "high"])
print("Равные интервалы:")
print(eq.value_counts().sort_index())

# По квартилям
q_bins = np.quantile(dept_A, [0, 0.25, 0.5, 0.75, 1.0])
q_cat = pd.cut(dept_A, bins=q_bins, labels=["Q1", "Q2", "Q3", "Q4"], include_lowest=True)
print("\nПо квартилям:")
print(q_cat.value_counts().sort_index())


---
## 2. Таблицы абсолютных частот

`value_counts()` считает, сколько наблюдений попало в каждую категорию.


In [ ]:
freq_A = cat_A.value_counts().sort_index()
freq_B = cat_B.value_counts().sort_index()

print("Частоты, отдел A:")
print(freq_A)
print("\nЧастоты, отдел B:")
print(freq_B)


---
## 3. Относительные частоты

Доля категории = частота / объём выборки  
(или `value_counts(normalize=True)`).


In [ ]:
rel_A = freq_A / len(dept_A)
rel_B = freq_B / len(dept_B)

print("Относительные частоты A:")
print(rel_A.round(3))
print("\nОтносительные частоты B:")
print(rel_B.round(3))


---
## 4. Столбчатые диаграммы по таблицам частот

Несколько графиков в одном окне: `plt.subplots(nrows, ncols)`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

freq_A.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Отдел A")
axes[0].set_ylabel("Частота")
axes[0].tick_params(axis="x", rotation=0)

freq_B.plot(kind="bar", ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Отдел B")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle("Возрастные группы: абсолютные частоты")
plt.tight_layout()
plt.show()


---
## 5. Объединение таблиц: `pd.concat`

Две (и более) серии частот можно сложить в одну таблицу по столбцам или по строкам.


In [ ]:
matrix = pd.concat([freq_A, freq_B], axis=1, keys=["Dept A", "Dept B"])
matrix = matrix.fillna(0).astype(int)
print(matrix)


### Stacked и grouped barplot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

matrix.plot(kind="bar", stacked=True, ax=axes[0], edgecolor="black")
axes[0].set_title("Stacked")
axes[0].set_ylabel("Частота")
axes[0].tick_params(axis="x", rotation=0)

matrix.plot(kind="bar", stacked=False, ax=axes[1], edgecolor="black")
axes[1].set_title("Grouped")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


---
## 6. Круговые диаграммы

`plt.pie` показывает структуру как доли целого.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]

axes[0].pie(freq_A, labels=freq_A.index, autopct="%1.1f%%",
            colors=colors, startangle=90)
axes[0].set_title("Отдел A")

axes[1].pie(freq_B, labels=freq_B.index, autopct="%1.1f%%",
            colors=colors, startangle=90)
axes[1].set_title("Отдел B")

plt.suptitle("Структура возрастных групп")
plt.tight_layout()
plt.show()


---
## 7. Сравнение относительных частот двух групп


In [ ]:
compare = pd.DataFrame({"Dept A": rel_A, "Dept B": rel_B}).fillna(0)
print(compare.round(3))

compare.plot(kind="bar", figsize=(8, 4), edgecolor="black")
plt.ylabel("Относительная частота")
plt.xlabel("Группа")
plt.title("Сравнение структуры возрастов")
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Разбить на интервалы | `pd.cut(x, bins=..., labels=..., include_lowest=True)` |
| Абсолютные частоты | `series.value_counts()` |
| Относительные частоты | `series.value_counts(normalize=True)` или `counts / n` |
| Склеить столбцы | `pd.concat([a, b], axis=1)` |
| Склеить строки | `pd.concat([a, b], axis=0)` |
| Столбчатая диаграмма | `series.plot(kind="bar")` |
| Stacked / grouped | `df.plot(kind="bar", stacked=True/False)` |
| Круговая | `plt.pie(values, labels=..., autopct="%1.1f%%")` |
| Несколько графиков | `plt.subplots(nrows, ncols)` |

---
## Что сделать после лекции

1. Повторите `pd.cut` с **другими** границами и подписями.
2. Откройте лабораторное задание и выполните его **самостоятельно** (свои массивы, свои шкалы оценок).
3. Следите за `include_lowest` и порядком границ в `bins`, иначе появятся `NaN` или «пустые» категории.

Удачи с таблицами частот!
